In [1]:
from sequana import DNA
from sequana import FastA
import pandas as pd
import numpy as np
from sklearn.preprocessing import  StandardScaler
import re
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
from scipy.signal import argrelextrema
from itertools import combinations

from scipy.signal import peak_widths


In [2]:
def count_homopolymers(seq, min_length=5):
    # Ex : trouve AAAAA ou TTTTT, etc.
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def load_fasta(fasta_path, window_size=100):
    f = FastA(fasta_path)
    data = []


    for maseq in f:

        print(maseq.name)
        
        features = []

        s = DNA(maseq.sequence.upper())
        seq = maseq.sequence.upper()
        s.window = window_size

       

        #Homopolymere
        X2 = []
        X3 = []

        
        for i in range(0, len(seq)-2, 1):
            
            window = seq[max(0, i - window_size//2):min(i+window_size//2,len(seq))]
            nb = count_homopolymers(window, min_length=2)
            X2.append(nb)

            nb = count_homopolymers(window, min_length=3)
            X3.append(nb)
        


            
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]


 






   

        df= pd.DataFrame({
            'X2':X2,
            'X3':X3
        })

        
        data.append(df)

            
    return data

 
data = load_fasta("../data/Fasta/TriTrypDB-68_LinfantumJPCM5_Genome.fasta",200)


LinJ.01
LinJ.02
LinJ.03
LinJ.04
LinJ.05
LinJ.06
LinJ.07
LinJ.08
LinJ.09
LinJ.10
LinJ.11
LinJ.12
LinJ.13
LinJ.14
LinJ.15
LinJ.16
LinJ.17
LinJ.18
LinJ.19
LinJ.20
LinJ.21
LinJ.22
LinJ.23
LinJ.24
LinJ.25
LinJ.26
LinJ.27
LinJ.28
LinJ.29
LinJ.30
LinJ.31
LinJ.32
LinJ.33
LinJ.34
LinJ.35
LinJ.36


In [4]:
df = pd.read_csv("../data/Centromere_Positions/pos_libre.csv")
pos_libre = {
    int(row.Chromosome): (int(row.Start), int(row.End))
    for row in df.itertuples(index=False)
}

vecteur_Debut = []
vecteur_Fin = []

vecteur_longeur = []


g = 0


for i in range(0,36):
    temp_ =  data[i]['X3']*data[i]['X2']
    temp_ = temp_[40000:-5000]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 



        #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    
  #  for peak in peaks:
  #      start = max(0, peak - window//2)
  #      end = min(len(temp_), peak + window//2 + 1)
    
   #     neighborhood = temp_[start:end]
   #     if len(neighborhood) > 0:
   #         median_val = np.median(neighborhood)
   #         peak_medians.append((peak, median_val))
    
    # Trouver le pic avec la médiane la plus élevée
    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])
        max_index = peak_max


        #Chercher la taille du centromere


        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        temp_X2_3 = temp_X2_3[max_index-25000:max_index+25000]
        

        
        # Détection des pics
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)

            
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.4)
        
            taille = widths_result[0][peak_index]
            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]
            seuil =  widths_result[1][peak_index]



            debut = int(debut)
            fin = int(fin)
            
            marge = 750
            finish = True
            distance = []
            temp_fin = []
            while finish:
                
                recherche = False
                limiteFin = min(len(temp_X2_3), fin + marge)
                pos = fin

                while pos < limiteFin and recherche == False:
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos += 1
                if recherche == True:
                     temp_fin.append(fin)
                     distance.append(0)
                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos += 1
                        distance[len(distance)-1] += 1

                     fin = pos
                else: 
                    finish = False

            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                fin = temp_fin[t]
                t = t -1 
            #Avant
            finish = True
            distance = 0
            distance = []
            temp_debut = []

            while finish:
                recherche = False
                limiteDebut = max(0, debut - marge)
                pos = debut
                while pos > limiteDebut and recherche == False:
 
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos -= 1
        
                if recherche == True:
                     temp_debut.append(debut)
                     distance.append(0)

                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos -= 1
                        distance[len(distance)-1] += 1
                     debut = pos
                else:
                    finish = False


            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                debut = temp_debut[t]
                t = t -1 
                

            
            
            

            debut = debut+max_index-25000+5000+40000
            fin = fin+max_index-25000+5000+40000
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+40000
            taille = fin - debut

            # Affichage



            
            pos_start, pos_end = pos_libre[i+1]

            if abs(pos_start-debut) > 50000:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille} Attention !!! Diff position : {pos_start-debut}')
            else:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')

   



            vecteur_Debut.append(debut)
            vecteur_Fin.append(fin)
            vecteur_longeur.append(taille)


print(len(vecteur_Debut))
result = pd.DataFrame()
result['Chromosome'] = list(range(1, 37))
result['start'] = vecteur_Debut 
result['end'] = vecteur_Fin
result['length'] = vecteur_longeur
result.to_csv(f"../output/estimation/infantum.csv", index=False)

1 : Commence 262107    Fini 263903  Taille : 1796
2 : Commence 269373    Fini 272783  Taille : 3410
3 : Commence 249506    Fini 255095  Taille : 5589
4 : Commence 119304    Fini 126028  Taille : 6724
5 : Commence 365567    Fini 372125  Taille : 6558
6 : Commence 123839    Fini 129118  Taille : 5279
7 : Commence 209911    Fini 215149  Taille : 5238
8 : Commence 444589    Fini 448932  Taille : 4343
9 : Commence 266948    Fini 271961  Taille : 5013
10 : Commence 326749    Fini 332008  Taille : 5259
11 : Commence 158820    Fini 161719  Taille : 2899
12 : Commence 284868    Fini 287961  Taille : 3093
13 : Commence 142650    Fini 145039  Taille : 2389
14 : Commence 168651    Fini 174251  Taille : 5600
15 : Commence 341990    Fini 347632  Taille : 5642
16 : Commence 333847    Fini 337385  Taille : 3538
17 : Commence 347380    Fini 351626  Taille : 4246
18 : Commence 434916    Fini 440803  Taille : 5887
19 : Commence 640456    Fini 646533  Taille : 6077
20 : Commence 514045    Fini 517756  Tai

In [6]:
print(len(data))

36
